In [ ]:
!pip install -r .\requirements.txt

# Modele

## Chat Model
Jeżeli nie masz pobranego modelu, wykonaj kod !ollama pul... aby pobrać model z .env

### Non-Streaming

In [ ]:
from app.core import LLM_MODEL
!ollama pull {LLM_MODEL}

In [ ]:
from app.core import ChatModel, LLM_MODEL

chat = ChatModel(model=LLM_MODEL, system="You are a helpful assistant", memory=False)
r = chat.ask("Napisz małą rozprawkę o Szekspirze", think=False)

print(r)

### Streaming
W .ipnyb streaming nie zadziała

In [ ]:
from app.core import ChatModel, LLM_MODEL

chat = ChatModel(model=LLM_MODEL, system="You are a helpful assistant", memory=False)
for piece in chat.ask_stream("Napisz małą rozprawkę o Szekspirze", think=False):
    print(piece, end="", flush=True)

## Embed Model
Jeżeli nie masz pobranego modelu, wykonaj kod !ollama pul... aby pobrać model z .env

In [ ]:
from app.core import EMBED_MODEL
!ollama pull {EMBED_MODEL}

In [ ]:
from app.core import EmbedModel, EMBED_MODEL, EMBED_MODEL_DIM

embed = EmbedModel(model=EMBED_MODEL, dim=EMBED_MODEL_DIM)
e = embed.encode("Lorem ipsum dolor sit amed")

print(e)

# RAG
Stwórz bazy danych tą komendą jeżeli jeszcze tego nie zrobiłeś

In [ ]:
!docker compose -f database/docker-compose.yml up -d

## Vector RAG

### Vector Database Set-up
Tworzymy baze danych w dockerze wykorzystując postgres i rozszerzenie pgvectors

In [ ]:
from app.schema import ProcedureStep

d = ProcedureStep(text='trorolorodwada')
print(d.model_dump(exclude_none=True))

## Graph RAG

In [ ]:
from app.core import GRAPH_MODEL
!ollama pull {GRAPH_MODEL}

### Graph Database Set-Up

In [4]:
from app.core import ChatModel, GRAPH_MODEL, langchain_tools_to_function_map, langchain_tools_to_ollama_format
from app.graph import *
import app.graph
from app.ingest import *
import app.ingest

app.ingest.knowledge_directory = Path("./knowledge")

knowledge_graph.clear()  # zawsze przed nowym testem

all_tools = [merge, relationship, status]

# lfm2.5 best
# gemma4:12b

chat = ChatModel(
    # model='lfm2.5',
    model='gemma4:12b',
    # model=GRAPH_MODEL,
    system="Jesteś asystentem budującym graf wiedzy służacy pomocą w obłudze ERP. Jedyne co cię interesuje to jak system operuje i ułożenie grafu tak, aby pomagał komuś kto nic nie rozumie. Przed tworzeniem relacji sprawdź dostępne id narzędziem status",
    tool_definitions=langchain_tools_to_ollama_format(all_tools),
    tools=langchain_tools_to_function_map(all_tools),
)

stri = ""

docs = load_knowledge()

for doc in docs:
    stri += app.schema.prepare_for_vector_embedding(document=doc)
    stri += "\n\n\n"

print(stri)

chat.pretty(stri)

nodes = app.graph.knowledge_graph.cypher("node")

print('\n'.join(nodes))
print("\n\n")
print('\n'.join(app.graph.knowledge_graph.cypher("relationship")))


Przyjęcie towaru na magazyn dokumentem PZ
Jak przyjąć towar od dostawcy na wybrany magazyn i zatwierdzić dokument.
jak przyjąć towar
towar przyjechał co teraz
jak zrobić pz
jak zwiększyć stan magazynowy
przyjęcie zewnętrzne


Wydanie towaru z magazynu dokumentem WZ
Jak wydać towar odbiorcy na zewnątrz i zmniejszyć stan magazynowy.
jak wydać towar
jak zrobić wz
jak zdjąć coś ze stanu
klient odbiera towar
wydanie zewnętrzne
Na magazynie źródłowym musi być wystarczająca ilość każdego produktu


Przesunięcie towaru między magazynami dokumentem MM
Jak przenieść towar z jednego magazynu na drugi. Magazyny muszą być różne, a zatwierdzić może tylko kierownik.
jak przesunąć towar między magazynami
jak zrobić mm
przesunięcie na produkcję
transfer między magazynami
Magazyn źródłowy i docelowy muszą być różne
Zatwierdzić MM może wyłącznie kierownik


Sprawdzenie stanu magazynowego produktu
Gdzie zobaczyć aktualne ilości produktów i co oznacza czerwone podświetlenie.
gdzie widzę stany
ile mam towar